# 🎵 Sound Metadata Extractor — Colab Launcher

This notebook **mounts Google Drive, installs dependencies, and runs the Python project**.

The actual code lives in the `Sound Meta Extractor/` folder on your Drive as a proper Python package.

---
### Steps
1. **Cell 1** — Mount Google Drive
2. **Cell 2** — Install system & Python dependencies
3. **Cell 3** — ⚙️ **Set your configuration** (paths, folder ID)
4. **Cell 4** — ▶️ Run the extractor
5. **Cell 5** — (Optional) Preview results in a table

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# ============================================================
# CELL 2 — Install Dependencies
# ============================================================

# System: ffmpeg for broad audio format support (AIFF, M4A, OGG, OPUS...)
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

# Python libraries (reads requirements.txt from your Drive project folder)
!pip install -q -r "/content/drive/MyDrive/Sound Meta Extractor/requirements.txt"
print('✅ All Python libraries installed')

In [ ]:
# ============================================================
# CELL 3 — ⚙️ CONFIGURATION  (edit these values before running)
# ============================================================
import os

# Full path to your sounds folder on Google Drive
# Example: '/content/drive/MyDrive/Sound Libraries/BLOW'
os.environ['SOUNDS_FOLDER'] = '/content/drive/MyDrive/SOUNDS'

# Google Drive Folder ID from the browser URL:
# https://drive.google.com/drive/folders/<PASTE_ID_HERE>
# Leave as empty string to skip Drive link generation.
os.environ['DRIVE_FOLDER_ID'] = ''  # e.g. '1AbCdEfGhIjKlMnOpQr'

# Where to save the output files (will be created automatically)
os.environ['OUTPUT_FOLDER'] = '/content/drive/MyDrive/sounds_metadata_output'

# Search inside subfolders? ('true' or 'false')
os.environ['RECURSIVE'] = 'true'

# Skip files larger than this in MB. Set '0' for no limit.
os.environ['MAX_FILE_SIZE_MB'] = '500'

print('✅ Configuration set:')
print(f'   SOUNDS_FOLDER   = {os.environ["SOUNDS_FOLDER"]}')
print(f'   DRIVE_FOLDER_ID = {os.environ["DRIVE_FOLDER_ID"] or "(not set — Drive links disabled)"}')
print(f'   OUTPUT_FOLDER   = {os.environ["OUTPUT_FOLDER"]}')
print(f'   RECURSIVE       = {os.environ["RECURSIVE"]}')

In [ ]:
# ============================================================
# CELL 4 — ▶️ Run the Extractor
# ============================================================
import sys

# Add project to Python path
PROJECT_PATH = '/content/drive/MyDrive/Sound Meta Extractor'
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

# Run main.py (inherits all environment variables set in Cell 3)
!cd "{PROJECT_PATH}" && python main.py

In [ ]:
# ============================================================
# CELL 5 — (Optional) Preview Results Table
# ============================================================
import json, os
import pandas as pd
from IPython.display import display

output_folder = os.environ.get('OUTPUT_FOLDER', '/content/drive/MyDrive/sounds_metadata_output')
json_path = f'{output_folder}/sounds.json'

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} records from {json_path}')

PREVIEW_COLS = [
    'filename', 'extension', 'sf_duration_seconds', 'sf_sample_rate',
    'sf_bit_depth', 'sf_channel_label', 'lb_tempo_bpm',
    'ai_top_class', 'ai_top_score', 'heuristic_sound_type',
    'fn_parsed_category', 'fn_parsed_description',
    'drive_preview_url',
]

def flatten(r):
    return {k: (str(v) if isinstance(v, (list, dict)) else v) for k, v in r.items()}

df = pd.DataFrame([flatten(r) for r in data])
available = [c for c in PREVIEW_COLS if c in df.columns]
display(df[available].head(20))